In [1]:
"""
kNN: koszt implementacji wyszukiwania sasiadow + weryfikacja rownowaznosci wynikow.

Panele = liczba cech, os x = k, kolor = implementacja, ksztalt = przejscie.
"""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from koszt_wspolne import (TRANSITIONS, TRANSITION_MARKERS, STYL, LINIA,
                           palety, rysuj_serie, legenda, zapisz)

DATA_DIR = "."               # katalog z plikami *_KNN_COMPARISON_summary.csv
WZORZEC = "{kod}_KNN_COMPARISON_summary.csv"
KOL_KRYT = {"AB": "Delta_crit", "AC": "K0_crit", "BCb": "Delta_crit"}

# Skan kNN liczono na innej probie niz przebieg glowny, a stosunek czasow nie
# jest staly (ok. 65-72x dla A--B, 2-4x dla A--C, niespojny dla B--C_b).
# Sekundy z tego rysunku nie sa wiec porownywalne z pozostalymi - pokazujemy
# czas wzgledny, bo i tak chodzi wylacznie o proporcje miedzy implementacjami.
NORMALIZACJA = "globalna"     # "na_przejscie" | "globalna" | "brak"

plt.rcParams.update(STYL)

# ---------------------------------------------------------------- dane
ramki = []
for kod in TRANSITIONS:
    d = pd.read_csv(f"{DATA_DIR}/SUPERVISED/{kod}/{WZORZEC.format(kod=kod)}")
    d = d.rename(columns={KOL_KRYT[kod]: "kryt",
                          KOL_KRYT[kod] + "_err": "kryt_err"})
    d["transition"] = kod
    ramki.append(d)
dane = pd.concat(ramki, ignore_index=True)

if NORMALIZACJA == "na_przejscie":
    odn = dane.groupby("transition")["runtime_model"].transform("max")
    dane["czas"] = dane["runtime_model"] / odn
    Y_OPIS = "czas względem najwolniejszej konfiguracji"
elif NORMALIZACJA == "globalna":
    dane["czas"] = dane["runtime_model"] / dane["runtime_model"].max()
    Y_OPIS = "czas względem najwolniejszej konfiguracji"
else:
    dane["czas"] = dane["runtime_model"]
    Y_OPIS = "czas modelu [s]"

# ================================================================
# 1. WERYFIKACJA: czy implementacje daja rozne wyniki?
# ================================================================
print("=" * 64)
print("Rozrzut wyniku po implementacjach (te same: cechy, k, przejscie)")
print("=" * 64)

g = dane.groupby(["transition", "n_features", "n_neighbors"])["kryt"]
rozrzut = (g.max() - g.min()).groupby(level=0).max()
warianty = g.nunique().groupby(level=0).max()

for kod in TRANSITIONS:
    print(f"  {TRANSITIONS[kod]:8s}  max|roznica| = {rozrzut[kod]:.3e}"
          f"   roznych wartosci = {warianty[kod]}")

if rozrzut.max() == 0:
    print("\n  -> wszystkie implementacje daja IDENTYCZNE wyniki (bitowo).")
    print("     Wybor implementacji jest wylacznie decyzja o kosztcie.")

# dryf po k, w jednostkach bledu statystycznego
print("\n" + "=" * 64)
print("Dryf wyniku przy zmianie k (1 -> 201)")
print("=" * 64)
for kod in TRANSITIONS:
    d = dane[dane.transition == kod]
    k = d.groupby("n_neighbors")["kryt"].mean()
    blad = d["kryt_err"].replace(0, np.nan).median()
    delta = k.loc[201] - k.loc[1]
    ile = abs(delta) / blad if blad == blad else np.nan
    print(f"  {TRANSITIONS[kod]:8s}  d = {delta:+.3e}"
          f"   sigma = {blad:.3e}   |d|/sigma = {ile:.2f}")

# ================================================================
# 2. RYSUNEK: czas w funkcji k
# ================================================================
CECHY = sorted(dane.n_features.unique(), reverse=True)

# kolor = przejscie fazowe, ksztalt/styl = implementacja
KOL_PRZEJSCIA = {"AB": "#4c72b0", "AC": "#c44e52", "BCb": "#55a868"}
ALG = {                       # marker, styl linii, rozmiar, wypelnienie
    "auto":      ("o", "-",  5.2, False),   # otwarte kolo, wieksze
    "brute":     ("D", "-",  2.8, True),    # wypelniony romb wpada w srodek 'auto'
    "kd_tree":   ("s", "--", 2.8, True),
    "ball_tree": ("^", ":",  2.8, True),
}

fig, axs = plt.subplots(1, len(CECHY), figsize=(7.4, 3.2),
                        sharex=True, sharey=True)

for ax, nf in zip(axs, CECHY):
    d_nf = dane[dane.n_features == nf]
    # 'auto' na wierzchu jako otwarty marker - wypelniony marker implementacji,
    # ktora zostala wybrana, jest wtedy widoczny w jego srodku
    for alg in ("brute", "kd_tree", "ball_tree", "auto"):
        marker, styl, ms, wypelniony = ALG[alg]
        for kod, kolor in KOL_PRZEJSCIA.items():
            d = d_nf[(d_nf.transition == kod)
                     & (d_nf.algorithm == alg)].sort_values("n_neighbors")
            if d.empty:
                continue
            ax.plot(d.n_neighbors, d["czas"],
                    color=kolor, ls=styl, lw=0.9, marker=marker, ms=ms,
                    markevery=2, alpha=0.95,
                    markerfacecolor=kolor if wypelniony else "none",
                    markeredgecolor=kolor,
                    markeredgewidth=0.9 if not wypelniony else 0.3,
                    zorder=4 if alg == "auto" else 3)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{nf} cech", loc="left", pad=3)
    ax.set_xlabel("liczba sąsiadów $k$")
    ax.grid(True, which="both", alpha=0.22, lw=0.5)

axs[0].set_ylabel(Y_OPIS)

for ax, nf in zip(axs, CECHY):
    sr = dane[dane.n_features == nf].groupby("algorithm").runtime_model.mean()
    blizszy = (sr.drop("auto") - sr["auto"]).abs().idxmin()
    ax.text(0.5, 0.03, f"auto $\\equiv$ {blizszy}", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=7, color="0.25",
            bbox=dict(fc="white", ec="0.8", lw=0.5, pad=2.2, alpha=0.92))

uchwyty = [Line2D([], [], color=k, lw=1.2, label=TRANSITIONS[t])
           for t, k in KOL_PRZEJSCIA.items()]
for alg, (marker, styl, ms, wyp) in ALG.items():
    uchwyty.append(Line2D([], [], color="0.35", ls=styl, lw=0.9, marker=marker,
                          ms=ms, label=alg,
                          markerfacecolor="0.35" if wyp else "none",
                          markeredgecolor="0.35",
                          markeredgewidth=0.9 if not wyp else 0.3))

fig.tight_layout(rect=[0, 0.13, 1, 1])
fig.legend(handles=uchwyty, loc="lower center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.005), handletextpad=0.5,
           columnspacing=1.3, handlelength=2.2)

zapisz(fig, "KNN_algorytmy_czas")

# ---------------------------------------------------------------- tabela
print("\n" + "=" * 64)
print("Sredni czas [s] (srednia po k)")
print("=" * 64)
tab = dane.pivot_table(index="algorithm", columns=["transition", "n_features"],
                       values="runtime_model", aggfunc="mean").round(2)
print(tab.to_string())

print("\nPrzyspieszenie kd_tree wzgledem auto:")
for kod in TRANSITIONS:
    for nf in CECHY:
        s = dane[(dane.transition == kod) & (dane.n_features == nf)]
        r = (s[s.algorithm == "auto"].runtime_model.mean()
             / s[s.algorithm == "kd_tree"].runtime_model.mean())
        print(f"  {TRANSITIONS[kod]:8s} {nf:>2d} cech: {r:5.1f}x")

Rozrzut wyniku po implementacjach (te same: cechy, k, przejscie)
  A--B      max|roznica| = 0.000e+00   roznych wartosci = 1
  A--C      max|roznica| = 0.000e+00   roznych wartosci = 1
  B--C$_b$  max|roznica| = 0.000e+00   roznych wartosci = 1

  -> wszystkie implementacje daja IDENTYCZNE wyniki (bitowo).
     Wybor implementacji jest wylacznie decyzja o kosztcie.

Dryf wyniku przy zmianie k (1 -> 201)
  A--B      d = +7.835e-06   sigma = 1.760e-06   |d|/sigma = 4.45
  A--C      d = +1.027e-03   sigma = 8.903e-04   |d|/sigma = 1.15
  B--C$_b$  d = -2.485e-05   sigma = 2.762e-05   |d|/sigma = 0.90

Sredni czas [s] (srednia po k)
transition    AB                 AC                  BCb              
n_features    12    20    30     12     20     30     12     20     30
algorithm                                                             
auto        0.29  2.62  2.71   2.09  22.45  21.49   0.48  10.57  10.80
ball_tree   0.50  0.51  0.68   7.41  11.25   7.45   1.08   1.39   1.33
brute   